# Experiment 0: verify the measurement pipeline

The zero-learning-rate controls exercise loading, forward/backward computation, optimizer plumbing, monitoring, saving and reloading while model weights must remain unchanged. There are 16 controls: four bases × two adaptation modes × two tasks, on one dataset partition.

A successful process exit is insufficient. Require the exact saved-state audit and per-dataset monitoring parity. These controls cannot validate positive-learning-rate optimization, long-run stability, recovery or final benchmark inference; those require the subsequent checks.

In [ ]:
%matplotlib inline
import sys
from pathlib import Path
REPO = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'pyproject.toml').is_file())
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
from IPython.display import display
from src.visualize import style
from src.visualize.figures import FigureSaver
from src.visualize import campaign as cp
from src.data.dataset_names import display_frame
from src.visualize.inputs import analysis_root
style.apply()
sink = FigureSaver('experiment0/01_null_controls')
report = cp.NotebookReport('Experiment 0: verify the measurement pipeline')


## 1. Planned and recorded controls

The config supplies the denominator, so absent trials appear as pending. An external analysis folder, if configured, is read only.

In [ ]:
controls = [cp.load_campaign(0, track, 'null') for track in ('pd','lgd')]
for control in controls:
    cp.show(sink, cp.plot_coverage(control))
report.add('1. Control coverage', '\n\n'.join(cp.coverage_summary(c) for c in controls))

## 2. Saved-checkpoint audits

These results come from completed CPU audit reports. The canonical model and inference buffers must match, and monitoring must match for every observed dataset. A missing audit is not a pass.

In [ ]:
audits = cp.null_audits(controls)
display(audits)
cp.show(sink, cp.plot_null_audits(audits))
report.add('2. Saved-state audits', audits.to_string(index=False) if len(audits) else 'No completed audit reports in the selected analysis source.')

## 3. Monitor invariance

Show the largest absolute dataset effect in each control, separately on training and held-out tables. The audit checks unrounded values; this figure makes departures from zero easy to locate.

In [ ]:
for control in controls:
    cp.show(sink, cp.plot_null_monitor(control))
report.add('3. Monitor invariance', '\n\n'.join(c.track.upper() + '\n' + cp.effect_summary(cp.endpoint_effects(c)) for c in controls))

## 4. What is still untested?

Before the main experiment, verify positive-LR behavior and interrupted/resumed execution on VSC. Then use the short and budget pilots to set resource requests and the training horizon. Two-update null timings are not production walltime estimates.

In [ ]:
report.add('4. Remaining gates', 'Saved-state equality is separate from positive-LR correctness. Next: positive-LR/recovery checks, short timing pilots, budget pilots, then settle the main horizon. No jobs are submitted by this notebook.')

## Summary

The following text repeats the sections in order. It is included verbatim in `All_Results.md`.

In [ ]:
print(report.summary(sink))